In [8]:
import pandas as pd
import numpy as np
import json
import glob
import csv
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models

In [9]:
def load_data(file):
    with open (file, "r", encoding="utf-8") as f:
        data = json.load(f)
    return (data)

def write_data(file, data):
    with open (file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [10]:
stopwords = stopwords.words("english")
stopwords.append("be")
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [11]:
# Load the JSON data from the file
with open("kenny_bag_of_words.json", "r") as file:
    data = json.load(file)

# Get the keys of the JSON object
fields = data.keys()

# Print the fields
print(fields)


dict_keys(['BagOfWords'])


In [12]:
data = load_data("kenny_bag_of_words.json")['BagOfWords']
print (data[0][200:590])
# print (data[1][0:90])

w,none,of,you,know,the,history,but,it’s,a,big,deal,i,took,bath,twice,for,this,is,that,even,possible,apparently,it,is,yes,the,royal,opera,house,the,great,deal,and,it’s,a,big,occasion,because,i’ve,just,turned,twentysix,yeah,i,have,i,don’t,know,why,people,cheer,for,that,because,they’re,like,“hey,he’s,going,to,die,soon,yaay”,yea,i,am,going,to,die,soon,yes,it’s,crazy,like,twentysix…,you,won’t


In [13]:
def lemmatization(texts, allowed_postages=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in data:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postages:
                new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return (texts_out)

lemmatized_texts = lemmatization(data)
print (lemmatized_texts[0][0:150])

thank so much mumbai thank thank really guy do come on guy royal opera house guy royal opera house big deal know none know history big deal take bath 


In [14]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [29]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'bro', 'give', 'bottle', 'happen', 'talk', 'want', 'nice', 'good', 'need', 'people', 'put', 'thing', 'look', 'write', 'make', 'take', 'subject', 're', 'edu', 'use', 'be', 'know', 'go', 'think', 'come', 'see', 'guy', 'say', 'even', 'year', 'one', 'would', 'find', 'get'])
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alisha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [30]:
def gen_words(data):
    final = []
    for text in data:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])
# print (data_words[1][0:20])

['thank', 'so', 'much', 'mumbai', 'thank', 'thank', 'really', 'guy', 'do', 'come', 'on', 'guy', 'royal', 'opera', 'house', 'guy', 'royal', 'opera', 'house', 'big']


In [31]:
data_words = remove_stopwords(data_words)
print(data_words[:1][0][:30])

['thank', 'much', 'mumbai', 'thank', 'thank', 'really', 'royal', 'opera', 'house', 'royal', 'opera', 'house', 'big', 'deal', 'none', 'history', 'big', 'deal', 'bath', 'twice', 'possible', 'apparently', 'royal', 'opera', 'house', 'great', 'deal', 'big', 'occasion', 'turn']


In [32]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=200)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0])

['thank', 'much', 'mumbai', 'thank', 'thank', 'really', 'royal', 'opera', 'house', 'royal', 'opera', 'house', 'big', 'deal', 'none', 'history', 'big', 'deal', 'bath', 'twice', 'possible', 'apparently', 'royal', 'opera', 'house', 'great', 'deal', 'big', 'occasion', 'turn', 'cheer', 'die', 'soon', 'die', 'soon', 'crazy', 'believe', 'kind', 'difficult', 'dance', 'much', 'dance', 'care', 'else', 'way', 'dance', 'dance', 'watch', 'apply', 'woman', 'great', 'dancing', 'man', 'stop', 'man', 'standing', 'dancing', 'enough', 'bar', 'music', 'dance', 'dancing', 'painful', 'watch', 'club', 'great', 'already', 'fun', 'already', 'high', 'great', 'spending', 'buck', 'drink', 'club', 'drink', 'way', 'drink', 'whatever', 'high', 'neat', 'neat', 'neat', 'neat', 'awesome', 'jack', 'coke', 'diet', 'diet', 'kind', 'barbaric', 'place', 'great', 'chance', 'still', 'high', 'shots', 'sir', 'kind', 'shots', 'sir', 'many', 'stop', 'shout', 'stop', 'care', 'like', 'shot', 'petroleum', 'petroleum', 'shot', 'bad',

In [33]:
# TF-IDF REMOVAL
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range (0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] # reinitialization to be safe, you can skip this
    tfidf_ids = [id for id, value in bow]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # the words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow
    

[(0, 1), (1, 3), (2, 2), (3, 2), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 2), (10, 1), (11, 2), (12, 1), (13, 1), (14, 1), (15, 5), (16, 9), (17, 1), (18, 6), (19, 1)]


In [34]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                           id2word=id2word,
                                           num_topics=3,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [35]:
lda_model.print_topics()

[(0,
  '0.002*"mom" + 0.002*"woman" + 0.002*"house" + 0.002*"home" + 0.001*"great" + 0.001*"friend" + 0.001*"time" + 0.001*"door" + 0.001*"tell" + 0.001*"guitar"'),
 (1,
  '0.001*"mom" + 0.001*"house" + 0.001*"woman" + 0.001*"home" + 0.001*"time" + 0.001*"great" + 0.001*"phone" + 0.001*"eye" + 0.001*"awesome" + 0.001*"song"'),
 (2,
  '0.011*"mom" + 0.008*"woman" + 0.008*"house" + 0.007*"home" + 0.006*"great" + 0.006*"time" + 0.005*"dance" + 0.005*"man" + 0.005*"die" + 0.005*"like"')]

In [37]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=15)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.027518  0.027399       1        1  99.931806
0      0.013550 -0.013441       2        1   0.034697
1      0.013967 -0.013958       3        1   0.033497, topic_info=       Term       Freq      Total Category  logprob  loglift
454     mom  21.000000  21.000000  Default  15.0000  15.0000
351   house  15.000000  15.000000  Default  14.0000  14.0000
789   woman  15.000000  15.000000  Default  13.0000  13.0000
346    home  13.000000  13.000000  Default  12.0000  12.0000
313   great  11.000000  11.000000  Default  11.0000  11.0000
..      ...        ...        ...      ...      ...      ...
244   enter   0.000848   8.185371   Topic3  -6.6226  -1.1732
629    shit   0.000851   9.061087   Topic3  -6.6197  -1.2720
572  reason   0.000845   7.307953   Topic3  -6.6261  -1.0634
205     die   0.000852   9.940638   Topic3  -6.6181  -1.3631
432     man   0.000850   9.941475   Topic3  -6.6201  -1.3651

[99 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
43        1  0.853376    asshole
54        1  0.977020    awesome
89        1  0.853787  boyfriend
124       1  0.853706      child
125       1  0.958288      chill
...     ...       ...        ...
743       1  0.854088      truck
748       1  0.853693    tsunami
764       1  0.852844       wake
772       1  0.993292      water
789       1  0.988189      woman

[62 rows x 3 columns], R=15, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 1, 2])

In [38]:
import re
# Load JSON data from file
with open('kenny_transcripts.json', 'r') as file:
    data = json.load(file)

# Extract transcripts from JSON data
transcripts = data.get('Transcripts', [])

# Iterate over each transcript
for transcript in transcripts:
    # Split the transcript into sentences
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', transcript)
    
    # Iterate over each sentence
    for sentence in sentences:
        # Check if 'friend' is in the sentence
        if 'water' in sentence.lower():
            print(sentence)

Guys, my eyes started watering.
It’s going to be about… This water bottle.
Did you even notice this water bottle?
No. Your friends ask for water, you put it in their mouth.
After you’re done with it, after it provides you with life nourishing water… What do you do?
This song is for the water bottle, the ultimate nice guy.
So this love song is for the water bottle.
Bottle, water bottle.
Bottle, water bottle.
 Bottle, water bottle.
Bottle, water bottle.” Want to make it more indie?
